# Fase 03: Estandarización a Capa Silver
**Objetivo:** Estandarizar, limpiar y normalizar los datos de Bronze aplicando transformaciones semánticas (Fahrenheit → Celsius, booleanos) y garantizar integridad referencial.
**Entradas:** Parquet desde `medicalproyect-raw/bronze/`.
**Salidas:** Parquet en `medicalproyect-processed/silver/` + logs de estandarización.


## 1. Inicialización y Carga desde Bronze


In [ ]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

# ── Función helper: sube el log al Data Lake ──
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass

import os
import logging
from datetime import datetime
from pyspark.sql.functions import col, trim, lower, upper, to_date, when, datediff, round
from pyspark.sql import SparkSession

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

log_filename = "pipeline_silver.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.Silver")

logger.info("=" * 50)
logger.info("INICIANDO PIPELINE DE ESTANDARIZACION (CAPA SILVER)")
logger.info("=" * 50)

spark = SparkSession.builder.appName("Medical_Silver_Layer").getOrCreate()
logger.info("Sesion Spark iniciada correctamente.")

STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW  = "medicalproyect-raw"

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"

CONTAINER_LOGS = "medicalproyect-logs"
base_path_bronze = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"

logger.info("Cargando datos desde la capa Bronze (Parquet)...")
df_patients    = spark.read.parquet(f"{base_path_bronze}/patients")
df_outcomes    = spark.read.parquet(f"{base_path_bronze}/outcomes")
df_medications = spark.read.parquet(f"{base_path_bronze}/medications")
df_lab_results = spark.read.parquet(f"{base_path_bronze}/lab_results")
df_diagnoses   = spark.read.parquet(f"{base_path_bronze}/diagnoses")
logger.info("Datos Bronze cargados correctamente.")

logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 03 Procesamiento Silver")


## 2. Estandarización Estructural, Semántica y Temporal


In [ ]:
# Estandarizamos los datos: ajustamos formatos de texto, convertimos unidades (F->C), normalizamos tipos y aplicamos reglas de negocio
logger.info("Aplicando estandarizacion Estructural, Semantica y Temporal...")

# =======================================================================
# 1. PATIENTS (Maestro)
# =======================================================================
df_patients_clean = df_patients \
    .withColumn("patient_id", trim(upper(col("patient_id")))) \
    .withColumn("sex", trim(upper(col("sex")))) \
    .withColumn("smoking_status", trim(lower(col("smoking_status")))) \
    .withColumn("insurance_type", trim(lower(col("insurance_type")))) \
    .withColumn("age", col("age").cast("integer")) \
    .withColumn("bmi", col("bmi").cast("double"))

# Conversion a Celsius (redondeo a 1 decimal)
df_patients_clean = df_patients_clean \
    .withColumn("temperature_c", round((col("temperature_f") - 32) * 5.0 / 9.0, 1)) \
    .drop("temperature_f") 

# Convertir dinamicamente todos los "dx_..." a Booleanos
columnas_dx = [c for c in df_patients_clean.columns if c.startswith("dx_")]
for c in columnas_dx:
    df_patients_clean = df_patients_clean.withColumn(c, col(c).cast("boolean"))

logger.info("   -> Tabla Patients limpiada (Fahrenheit a Celsius y Diagnosticos a Booleanos).")


# =======================================================================
# 2. OUTCOMES (Eventos)
# =======================================================================
df_outcomes_clean = df_outcomes \
    .withColumn("patient_id", trim(upper(col("patient_id")))) \
    .withColumn("discharge_disposition", trim(lower(col("discharge_disposition")))) \
    .withColumn("admission_date", to_date(col("admission_date"), "yyyy-MM-dd")) \
    .withColumn("discharge_date", to_date(col("discharge_date"), "yyyy-MM-dd")) \
    .withColumn("total_charges_usd", col("total_charges_usd").cast("double")) \
    .withColumn("icu_admission", col("icu_admission").cast("boolean")) \
    .withColumn("in_hospital_death", col("in_hospital_death").cast("boolean")) \
    .withColumn("readmitted_30d", col("readmitted_30d").cast("boolean"))
    
# Regla Temporal: Filtro de coherencia de fechas (se mantienen como DateType)
df_outcomes_clean = df_outcomes_clean.filter(col("discharge_date") >= col("admission_date"))

logger.info("   -> Tabla Outcomes limpiada (Filtro temporal aplicado, fechas como DateType).")


# =======================================================================
# 3, 4 y 5. MEDICATIONS, LAB_RESULTS y DIAGNOSES
# =======================================================================
df_medications_clean = df_medications \
    .withColumn("patient_id", trim(upper(col("patient_id")))) \
    .withColumn("medication", trim(lower(col("medication")))) \
    .withColumn("unit", trim(lower(col("unit")))) \
    .withColumn("dose", col("dose").cast("double")) \
    .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd")) \
    .withColumn("is_generic", col("is_generic").cast("boolean"))
logger.info("   -> Tabla Medications limpiada.")

df_lab_results_clean = df_lab_results \
    .withColumn("patient_id", trim(upper(col("patient_id")))) \
    .withColumn("test_name", trim(lower(col("test_name")))) \
    .withColumn("unit", trim(lower(col("unit")))) \
    .withColumn("value", col("value").cast("double")) \
    .withColumn("test_date", to_date(col("test_date"), "yyyy-MM-dd")) \
    .withColumn("is_abnormal", col("is_abnormal").cast("boolean"))
logger.info("   -> Tabla Lab Results limpiada.")

df_diagnoses_clean = df_diagnoses \
    .withColumn("patient_id", trim(upper(col("patient_id")))) \
    .withColumn("primary_diagnosis", trim(lower(col("primary_diagnosis")))) \
    .withColumn("primary_icd10", trim(upper(col("primary_icd10")))) \
    .withColumn("visit_date", to_date(col("visit_date"), "yyyy-MM-dd"))
logger.info("   -> Tabla Diagnoses limpiada.")
# Subida parcial de log tras esta seccion
_subir_log("silver/silver_integridad_referencial", STORAGE_ACCOUNT)


## 3. Integridad Referencial


In [ ]:
# Eliminamos duplicados en pacientes y filtramos registros huerfanos para mantener la integridad referencial
logger.info("Aplicando estandarizacion Relacional (Integridad Referencial)...")

# 1. Asegurar que no hay duplicados en el maestro de pacientes
df_patients_processed = df_patients_clean.dropDuplicates(["patient_id"])
logger.info(f"   -> Pacientes unicos garantizados: {df_patients_processed.count()} registros.")

# 2. Eliminar "Pacientes Huerfanos" (Left Semi Join)
df_outcomes_processed = df_outcomes_clean.join(df_patients_processed, "patient_id", "left_semi")
df_medications_processed = df_medications_clean.join(df_patients_processed, "patient_id", "left_semi")
df_lab_results_processed = df_lab_results_clean.join(df_patients_processed, "patient_id", "left_semi")
df_diagnoses_processed = df_diagnoses_clean.join(df_patients_processed, "patient_id", "left_semi")

logger.info("   -> Integridad referencial aplicada a las 4 tablas de hechos. Cero huerfanos.")
# Subida parcial de log tras esta seccion
_subir_log("silver/silver_escritura_en_capa_silver", STORAGE_ACCOUNT)


## 4. Escritura en Capa Silver


In [ ]:
# Guardamos los datos ya limpios y estandarizados en la capa Silver en formato Parquet
logger.info("Guardando datos limpios en Capa Silver (Formato Parquet)...")

# Apuntamos al contenedor de procesados bajo la carpeta silver
CONTAINER_PROCESSED = "medicalproyect-processed"
base_path_silver = f"abfss://{CONTAINER_PROCESSED}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

try:
    df_patients_processed.write.mode("overwrite").parquet(f"{base_path_silver}/patients")
    df_outcomes_processed.write.mode("overwrite").parquet(f"{base_path_silver}/outcomes")
    df_medications_processed.write.mode("overwrite").parquet(f"{base_path_silver}/medications")
    df_lab_results_processed.write.mode("overwrite").parquet(f"{base_path_silver}/lab_results")
    df_diagnoses_processed.write.mode("overwrite").parquet(f"{base_path_silver}/diagnoses")
    
    logger.info("Todos los DataFrames guardados en Parquet exitosamente.")
except Exception as e:
    logger.error(f"Error al guardar en Parquet: {e}")
# Subida parcial de log tras esta seccion
_subir_log("silver/silver_persistencia", STORAGE_ACCOUNT)


## 5. Persistencia de Logs y Cierre


In [ ]:
# Subimos el log de ejecucion al Data Lake y detenemos la sesion de Spark para liberar recursos
logger.info("Procediendo a guardar el archivo de logs en el Data Lake...")

# 1. APAGAR EL LOGGER RAIZ (ROOT)
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

try:
    from notebookutils import mssparkutils
    
    # 2. BUSCAR RUTA REAL Y RUTA DESTINO
    ruta_local_absoluta = "file://" + os.path.abspath(log_filename)
    
    fecha_str = datetime.now().strftime('%Y%m%d_%H%M')
    ruta_destino_log = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/silver_{fecha_str}.log"
    
    # 3. COPIAR AL DATA LAKE
    mssparkutils.fs.cp(ruta_local_absoluta, ruta_destino_log)
    
    print(f"EXITO: Logs de procesamiento guardados permanentemente en: {ruta_destino_log}")
    print("==================================================")
    print("PIPELINE SILVER FINALIZADO")
    print("==================================================")
    
except Exception as e:
    print(f"ERROR AL MOVER LOGS: No se pudieron copiar al Data Lake. Detalle: {e}")

# 4. Detener la sesion de Spark
logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 03 Procesamiento Silver")
spark.stop()